In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score


LOAD DATA

In [2]:
df = pd.read_csv(r"D:\ML\SM Modeling\Data\Raw\incident_enriched.csv")

df['short_description'] = df['short_description'].fillna('')
df['close_notes'] = df['close_notes'].fillna('')
df['task_text'] = df['task_text'].fillna('')
df['close_code'] = df['close_code'].fillna('')
df['u_resolution_sub_category'] = df['u_resolution_sub_category'].fillna('')

df["text"] = (
    df["short_description"] + " " +
    df["close_notes"] + " " +
    df["task_text"]
)

# 🔥 add resolution context
df["text_with_res"] = df["text"] + " RESOLUTION_" + df["close_code"]

# remove empty subcategory
df = df[df["u_resolution_sub_category"] != ""].reset_index(drop=True)

print("Rows:", df.shape)
print("Unique subcategories:", df["u_resolution_sub_category"].nunique())


# add resolution into input
df["text_with_res"] = df["text"] + " RESOLUTION_" + df["close_code"]

df = df.dropna(subset=["u_resolution_sub_category"])
df = df.reset_index(drop=True)

print(df.shape)


Rows: (85463, 17)
Unique subcategories: 78
(85463, 17)


REMOVE VERY RARE SUBCATEGORIES

In [3]:
counts = df["u_resolution_sub_category"].value_counts()

valid = counts[counts >= 20].index
df = df[df["u_resolution_sub_category"].isin(valid)].reset_index(drop=True)

print("After cleaning:", df.shape)
print("Remaining subcategories:", df["u_resolution_sub_category"].nunique())


After cleaning: (85370, 17)
Remaining subcategories: 59


TRAIN TEST SPLIT

In [4]:
from sklearn.model_selection import train_test_split

X = df["text_with_res"]
y = df["u_resolution_sub_category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(len(X_train), len(X_test))


68296 17074


TFIDF

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=80000,
    ngram_range=(1,2),
    stop_words='english',
    min_df=3,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape)


(68296, 80000)


SVD

In [6]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=500, random_state=42)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)


TRAIN MODEL

In [7]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

model_sub = SGDClassifier(
    loss='log_loss',
    max_iter=3000,
    n_jobs=-1,
    class_weight='balanced'
)

model_sub.fit(X_train_svd, y_train)

pred = model_sub.predict(X_test_svd)
acc = accuracy_score(y_test, pred)

print("\n🎯 Subcategory Accuracy:", round(acc*100,2), "%")



🎯 Subcategory Accuracy: 68.14 %


ADD CONFIDENCE ANALYSIS

In [8]:
import numpy as np
import pandas as pd

probs = model_sub.predict_proba(X_test_svd)
top1 = np.max(probs, axis=1)
pred_labels = model_sub.predict(X_test_svd)

result_df = pd.DataFrame({
    "actual": y_test.values,
    "pred": pred_labels,
    "confidence": top1
})

result_df["correct"] = result_df["actual"] == result_df["pred"]


CONFIDENCE BUCKET ANALYSIS

In [9]:
bins = [0,0.5,0.6,0.7,0.8,0.9,1.0]

result_df["bucket"] = pd.cut(result_df["confidence"], bins)

report = result_df.groupby("bucket").agg(
    total=("correct","count"),
    accuracy=("correct","mean")
).reset_index()

print(report)


       bucket  total  accuracy
0  (0.0, 0.5]  14686  0.632848
1  (0.5, 0.6]    648  0.964506
2  (0.6, 0.7]   1392  0.984195
3  (0.7, 0.8]    347  0.991354
4  (0.8, 0.9]      1  1.000000
5  (0.9, 1.0]      0       NaN


C:\Users\anshuman.r\AppData\Local\Temp\ipykernel_35512\3264967096.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  report = result_df.groupby("bucket").agg(


In [10]:
import joblib, os
save_dir = r"D:\ML\SM Modeling\AI_ENGINE_V3\models"
os.makedirs(save_dir, exist_ok=True)

joblib.dump(model_sub, save_dir + r"\sub_model.pkl")
joblib.dump(tfidf, save_dir + r"\sub_tfidf.pkl")
joblib.dump(svd, save_dir + r"\sub_svd.pkl")

print("Subcategory model saved")


Subcategory model saved
